# Resolve exact geolocation inputs for the unlinked Q3 case

Verify the newly downloaded NOAA-20 L2 file, inspect its actual mask schema and class counts, find the matched Aqua source L2 file already in the 39-file sample, and extract both exact geolocation inputs from L2 metadata. Resolve those exact filenames through CMR and write a bounded two-file plan. This notebook does not infer a geolocation filename from time or naming patterns and does not download geolocation data.

In [1]:
from pathlib import Path
import sys,json,hashlib
import pandas as pd
import h5py
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/reference').is_dir())
sys.path.insert(0,str(ROOT))
from src.firewatch.coverage_validation import sha256_file
from src.firewatch.geolocation_manifest import extract_geolocation_reference,query_exact_geolocation_manifest
sample=pd.read_csv(ROOT/'data/processed/source_pixel_validation_2025_stratified.csv')
plan=pd.read_csv(ROOT/'data/reference/unlinked_opportunity_l2_plan_2025.csv')
aqua=sample[sample.observation_id.eq(plan.iloc[0].source_observation_id)].iloc[0]
aqua_l2=ROOT/'data/reference/granules/validation_2025'/aqua.granule_id
n20_l2=ROOT/'data/reference/granules/unlinked_opportunity_pilot'/(str(plan.iloc[0].producer_granule_id)+'.nc')
# Verify the new plan and downloaded NOAA-20 L2 bytes.
n20_manifest=json.loads((n20_l2.parent/'download_manifest.json').read_text(encoding='utf-8'))
n20_record=n20_manifest['downloads'][0]
assert n20_manifest['plan_sha256']==hashlib.sha256((ROOT/'data/reference/unlinked_opportunity_l2_plan_2025.csv').read_bytes()).hexdigest()
assert n20_l2.is_file() and n20_l2.stat().st_size==n20_record['bytes'] and sha256_file(n20_l2)==n20_record['sha256']
# Verify Aqua L2 against the existing 39-file manifest.
l2_manifest=json.loads((aqua_l2.parent/'download_manifest.json').read_text(encoding='utf-8'))
aqua_record=next(x for x in l2_manifest['downloads'] if x['filename']==aqua_l2.name)
assert aqua_l2.stat().st_size==aqua_record['bytes'] and sha256_file(aqua_l2)==aqua_record['sha256']
display(pd.DataFrame([{'sensor':'Aqua','L2':aqua_l2.name,'bytes':aqua_l2.stat().st_size,'sha256':aqua_record['sha256']},{'sensor':'NOAA-20','L2':n20_l2.name,'bytes':n20_l2.stat().st_size,'sha256':n20_record['sha256']}]))

,sensor,L2,bytes,sha256
0,Aqua,MYD14.A2025261.1405.061.2025262112616.hdf,406512,cb9167550070de19375cc0ba1fce2d66e62e3672becdcf...
1,NOAA-20,VJ114IMG.A2025261.1330.002.2025261205555.nc,2836999,b23479ae7b10861151db2e2a8d7cabaf35f497245e2361...


In [2]:
import numpy as np
with h5py.File(n20_l2,'r') as f:
    mask=f['fire mask']
    counts=np.bincount(mask[:].ravel(),minlength=10)
    legend=mask.attrs.get('legend')
    fire_pixel_records=int(f['FP_latitude'].shape[0])
    mask_shape=mask.shape
print('NOAA-20 fire-mask shape:',mask_shape)
print('Active-fire records:',fire_pixel_records)
display(pd.DataFrame([{'mask_class':k,'pixel_count':int(v)} for k,v in enumerate(counts[:10])]))
print('Mask legend metadata:',legend)

NOAA-20 fire-mask shape: (6464, 6400)
Active-fire records: 16


,mask_class,pixel_count
0,0,2080
1,1,5326336
2,2,2637588
3,3,4310442
4,4,9583401
5,5,19509735
6,6,2
7,7,0
8,8,16
9,9,0


Mask legend metadata: [b'0 not-processed (non-zero QF)\n1 bowtie\n2 glint\n3 water\n4 clouds\n5 clear land\n6 unclassified pixel\n7 low confidence fire pixel\n8 nominal confidence fire pixel\n9 high confidence fire pixel']


In [3]:
from urllib.parse import urlencode
from urllib.request import urlopen
from src.firewatch.geolocation_manifest import CMR_GRANULES_URL

aqua_ref = extract_geolocation_reference(aqua_l2)
aqua_ref.update({
    'case_context': 'Aqua no-candidate source observation',
    'requested_geolocation_granule_id': aqua_ref['geolocation_granule_id'],
    'cmr_lineage_status': 'exact L2 input filename found in CMR',
})
n20_ref = extract_geolocation_reference(n20_l2)
requested_n20_geo = n20_ref['geolocation_granule_id']
# First test whether CMR contains the exact input named by the N20 L2 file.
try:
    query_exact_geolocation_manifest([n20_ref])
    exact_n20_found = True
except ValueError as exc:
    if 'CMR did not return' not in str(exc):
        raise
    exact_n20_found = False

# If the exact processing instance is absent, find a CMR-listed reprocessing
# with the same product, acquisition day/time, and collection segment.
query_params = [
    ('short_name', n20_ref['geolocation_short_name']),
    ('version', n20_ref['cmr_version']),
    ('temporal', f"{plan.iloc[0].granule_start_utc},{plan.iloc[0].granule_end_utc}"),
    ('page_size', '100'),
]
query_url = CMR_GRANULES_URL + '?' + urlencode(query_params)
with urlopen(query_url, timeout=60) as response:
    alternatives = json.load(response).get('feed', {}).get('entry', [])
prefix = '.'.join(requested_n20_geo.split('.')[:4]) + '.'
same_acquisition = [entry for entry in alternatives
                    if entry.get('producer_granule_id', '').startswith(prefix)]
if exact_n20_found:
    n20_plan_ref = {**n20_ref,
        'requested_geolocation_granule_id': requested_n20_geo,
        'cmr_lineage_status': 'exact L2 input filename found in CMR'}
else:
    if not same_acquisition:
        raise ValueError(f'Exact NOAA-20 geolocation is missing and CMR has no same-acquisition reprocessing: {requested_n20_geo}')
    same_acquisition.sort(key=lambda entry: (entry.get('updated', ''), entry.get('producer_granule_id', '')))
    replacement_name = same_acquisition[-1]['producer_granule_id']
    n20_plan_ref = {
        **n20_ref,
        'requested_geolocation_granule_id': requested_n20_geo,
        'geolocation_granule_id': replacement_name,
        'cmr_lineage_status': 'reprocessed same acquisition; exact input filename absent from CMR; must validate against L2 FP line/sample coordinates',
    }
n20_plan_ref['case_context'] = 'NOAA-20 overpass paired with Aqua no-candidate observation'
geo_rows, metadata = query_exact_geolocation_manifest([aqua_ref, n20_plan_ref])
metadata['original_n20_l2_geolocation_input'] = requested_n20_geo
metadata['original_n20_exact_cmr_match'] = exact_n20_found
metadata['same_acquisition_reprocessing_candidates'] = [
    entry.get('producer_granule_id') for entry in same_acquisition
]
metadata['lineage_caveat'] = (
    'If a reprocessed NOAA-20 geolocation file is used, its producer processing timestamp differs from the L2 input named by VJ114IMG. '
    'Validate its coordinates at the L2 FP_line/FP_sample indices before using it for cross-swath coverage.'
)
geo_plan = pd.DataFrame(geo_rows)
out = ROOT / 'data/reference/unlinked_opportunity_geolocation_plan_2025.csv'
geo_plan.to_csv(out, index=False)
out.with_suffix('.metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
display(geo_plan[['case_context','source_l2_filename','requested_geolocation_granule_id','geolocation_granule_id','cmr_lineage_status','cmr_collection_version','granule_size_cmr_mb']])
print(f'Combined CMR size estimate: {geo_plan.granule_size_cmr_mb.sum():.3f} MiB; no geolocation data downloaded.')


,case_context,source_l2_filename,requested_geolocation_granule_id,geolocation_granule_id,cmr_lineage_status,cmr_collection_version,granule_size_cmr_mb
0,Aqua no-candidate source observation,MYD14.A2025261.1405.061.2025262112616.hdf,MYD03.A2025261.1405.061.2025262150419.hdf,MYD03.A2025261.1405.061.2025262150419.hdf,exact L2 input filename found in CMR,6.1,31.719399
1,NOAA-20 overpass paired with Aqua no-candidate...,VJ114IMG.A2025261.1330.002.2025261205555.nc,VJ103IMG.A2025261.1330.021.2025261182205.nc,VJ103IMG.A2025261.1330.021.2025267192836.nc,reprocessed same acquisition; exact input file...,2.1,182.364589


Combined CMR size estimate: 214.084 MiB; no geolocation data downloaded.


## Interpretation boundary

The NOAA-20 Level-2 file's spatial grid exists for the full swath even if a target has no active-fire record. Its mask will later indicate whether the nearest geolocated pixel is clear land, cloud, water, fire, not processed, or unknown. The present mask histogram is whole-granule context only; without the paired geolocation companion, it cannot classify the Aqua coordinate. Likewise, a CMR exact geolocation record establishes product lineage and availability, not valid pixel coverage.